In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "../data/c_a/raw"
OUT_DIR = "../data/c_a/processed"
os.makedirs(OUT_DIR, exist_ok=True)

K0_PHASE_C = 4.67
K0_PHASE_A = 4.93
K0_EPS = 1e-6
DOWNSAMPLE = 20

In [3]:
def parse_nso(line: str):
    parts = line.split()
    idx = parts.index("Nso")
    after = parts[2:]  # skip "# Nso"

    return [
        float(after[0]),  # 16316
        float(after[3]),  # 
        float(after[5]),  # 1.178870
    ]

# def parse_nso(line: str):
 
#     # split line into tokens
#     parts = line.split()

#     # find the position of the 'Nso' keyword
#     idx = parts.index("Nso")

#     # take everything after 'Nso'
#     after = parts[idx + 1:]

#     values = after[0] + after[2] + after[5]

#     return [float(v) for v in values]


def parse_k0_from_filename(filename: str) -> float:
    """
    Extract the coupling constant κ₀ from the filename.

    Example filename:
        vto-4.67-0.6-T4-100k-torus-L.out

    The second '-' separated field encodes κ₀.

    Returns:
        float: κ₀ value
    """
    return float(filename.split("-")[1])


def flatten_sample(sample):
    return list(sample["Nso"])


def label_from_k0(k0):
    """
    Assign a phase label based on κ₀.

    Deep inside phase C:
        label = 0
    Deep inside phase A:
        label = 1
    Intermediate κ₀ values:
        label = None (not used for training)

    A small tolerance is used to avoid floating-point issues.

    Returns:
        int or None
    """
    if abs(k0 - K0_PHASE_C) < K0_EPS:
        return 0
    if abs(k0 - K0_PHASE_A) < K0_EPS:
        return 1
    return None

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
files

['vto-4.67-0.6-T4-100k-torus-L.out',
 'vto-4.70-0.6-T4-100k-torus-L.out',
 'vto-4.72-0.6-T4-100k-torus-L.out',
 'vto-4.73-0.6-T4-100k-torus-L.out',
 'vto-4.74-0.6-T4-100k-torus-L.out',
 'vto-4.75-0.6-T4-100k-torus-L.out',
 'vto-4.76-0.6-T4-100k-torus-L.out',
 'vto-4.77-0.6-T4-100k-torus-L.out',
 'vto-4.78-0.6-T4-100k-torus-L.out',
 'vto-4.79-0.6-T4-100k-torus-L.out',
 'vto-4.81-0.6-T4-100k-torus-L.out',
 'vto-4.82-0.6-T4-100k-torus-LL.out',
 'vto-4.83-0.6-T4-100k-torus-LL.out',
 'vto-4.84-0.6-T4-100k-torus-L.out',
 'vto-4.85-0.6-T4-100k-torus-LL.out',
 'vto-4.86-0.6-T4-100k-torus-LL.out',
 'vto-4.87-0.6-T4-100k-torus-LL.out',
 'vto-4.88-0.6-T4-100k-torus-LL.out',
 'vto-4.90-0.6-T4-100k-torus-LL.out',
 'vto-4.93-0.6-T4-100k-torus-LL.out']

In [6]:
X_full = []
y_full = []
K0_full = []

for file_idx, filename in enumerate(tqdm(files, desc="Parsing files")):

    if filename == 'vto-4.77-0.6-T4-100k-torus-L.out':
        SKIP_SAMPLES = 350_000 
    #if file_idx in (0, len(files) - 1) else 400_000
    
        file_k0 = parse_k0_from_filename(filename)
        file_label = label_from_k0(file_k0)
    
        file_path = os.path.join(DATA_DIR, filename)
    
        current_sample = None
        sample_counter = 0
    
        with open(file_path, "r") as f:
            for line in f:
                line = line.strip()
    
                if line.startswith("# ntime"):
    
                    # ---- SAVE previous sample
                    if current_sample is not None:
                        sample_counter += 1
    
                        if sample_counter > SKIP_SAMPLES:
                            X_full.append(flatten_sample(current_sample))
                            K0_full.append(file_k0)
                            y_full.append(file_label)
    
                    # ---- START new sample (THIS WAS WRONG BEFORE)
                    current_sample = {
                        "Nso": None,
                    }
    
                elif line.startswith("# Nso"):
                    if current_sample is not None:
                        current_sample["Nso"] = parse_nso(line)
    
        # ---- SAVE LAST SAMPLE (important!)
        if current_sample is not None:
            sample_counter += 1
            if sample_counter > SKIP_SAMPLES:
                X_full.append(flatten_sample(current_sample))
                K0_full.append(file_k0)
                y_full.append(file_label)
    
        print(f"{filename}: parsed {sample_counter - SKIP_SAMPLES} samples")



Parsing files: 100%|████████████████████████████| 20/20 [00:02<00:00,  8.55it/s]

vto-4.77-0.6-T4-100k-torus-L.out: parsed 119522 samples


In [7]:
X_full = np.asarray(X_full)
K0_full = np.asarray(K0_full)
y_full = np.asarray(y_full, dtype=object)

np.savez(
    os.path.join(OUT_DIR, "R_dataset_full_477.npz"),
    X=X_full,
    y=y_full,
    K0=K0_full
)

print("R_dataset_full_477.npz zapisany")
print("X_full shape:", X_full.shape)


R_dataset_full_477.npz zapisany
X_full shape: (119522, 3)


In [8]:
# X_full = np.asarray(X_full)
# K0_full = np.asarray(K0_full)
# y_full = np.asarray(y_full, dtype=object)

# np.savez(
#     os.path.join(OUT_DIR, "R_dataset_reweight_full.npz"),
#     X=X_full,
#     y=y_full,
#     K0=K0_full
# )

# print("dataset_reweight_full.npz zapisany")
# print("X_full shape:", X_full.shape)


In [9]:
# mask_train = (
#     (K0_full == K0_PHASE_C) |
#     (K0_full == K0_PHASE_A)
# ) & (y_full != None)

# X_train = X_full[mask_train]
# y_train = y_full[mask_train].astype(int)
# K0_train = K0_full[mask_train]

# # downsampling
# idx = np.arange(len(X_train)) % DOWNSAMPLE == 0

# X_train = X_train[idx]
# y_train = y_train[idx]
# K0_train = K0_train[idx]

# np.savez(
#     os.path.join(OUT_DIR, "R_dataset_train_reweight_shortened.npz"),
#     X=X_train,
#     y=y_train,
#     K0=K0_train
# )

# print("dataset_train_reweight_shortened.npz zapisany")
# print("X_train shape:", X_train.shape)
